# CIFAR-10-like Dataset Evaluation and Structuring
This notebook does the steps in the requested order:
1. **Calculate FID and Inception Score first** on the dataset in its current class-folder layout.
2. **Split** the dataset into CIFAR-10-style `train/` and `test/` folders with a 50k/10k split.
3. **Zip** the final structured dataset.


In [1]:
!unzip cifar32.zip
!pip install torch-fidelity

unzip:  cannot find or open cifar32.zip, cifar32.zip.zip or cifar32.zip.ZIP.


In [2]:

# ===== 1) Setup =====
# Adjust these paths before running.

SYNTH_ROOT = "/workspace/cifar32"   # current dataset root with 10 class folders
OUT_ROOT = "CIFAR10_like_structured"
ZIP_PATH = "CIFAR10_like_structured.zip"

# Reproducibility
SEED = 42

# Image size for FID / IS computation
# torch-fidelity uses InceptionV3 internally and can handle CIFAR-size images,
# but keeping images as-is is fine here since the library handles preprocessing.
BATCH_SIZE = 150
NUM_WORKERS = 4
DEVICE = "cuda"


In [3]:

# ===== 2) Install / import =====
# Run this cell once if needed.

# %pip install -q torch torchvision torch-fidelity pillow tqdm

import os
import random
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict

import torch
from torchvision import datasets
from tqdm.auto import tqdm

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA GPU not available. This notebook is intended to run on GPU.")


torch: 2.11.0+cu130
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [4]:
Path("/workspace/cifar32").exists()

True

In [5]:

# ===== 3) Validate dataset layout =====

CLASS_NAMES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

synth_root = Path(SYNTH_ROOT)
assert synth_root.exists(), f"Dataset root not found: {synth_root}"

present = sorted([p.name for p in synth_root.iterdir() if p.is_dir()])
print("Found class folders:", present)

missing = [c for c in CLASS_NAMES if c not in present]
extra = [c for c in present if c not in CLASS_NAMES]

if missing:
    raise ValueError(f"Missing CIFAR-10 classes: {missing}")
if extra:
    print("Warning: extra folders found:", extra)

counts = {}
total = 0
for c in CLASS_NAMES:
    n = len([p for p in (synth_root / c).iterdir() if p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".webp"}])
    counts[c] = n
    total += n

print("Per-class counts:")
for k,v in counts.items():
    print(f"  {k:>10}: {v}")
print("Total:", total)

assert total == 60000, f"Expected 60000 samples, found {total}"


Found class folders: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Per-class counts:
    airplane: 6000
  automobile: 6000
        bird: 6000
         cat: 6000
        deer: 6000
         dog: 6000
        frog: 6000
       horse: 6000
        ship: 6000
       truck: 6000
Total: 60000


In [6]:

# ===== 4) Compute FID and Inception Score FIRST =====
# We compare:
#   - real dataset reference: original CIFAR-10 train split (50k real images)
#   - generated dataset: your current SYNTH_ROOT folder structure
#
# FID: lower is better
# Inception Score: higher is generally better (but interpret with care)

from torch_fidelity import calculate_metrics

# Download CIFAR-10 reference dataset if needed
cifar_ref_root = Path("./cifar10_reference_data")
cifar_ref_root.mkdir(parents=True, exist_ok=True)

_ = datasets.CIFAR10(root=str(cifar_ref_root), train=True, download=True)

real_path = cifar_ref_root / "cifar-10-batches-py"
print("CIFAR-10 downloaded under:", cifar_ref_root)
print("Synthetic dataset path:", synth_root)

# torch-fidelity supports torchvision datasets by name.
# For the synthetic dataset, an image folder path works directly.
metrics = calculate_metrics(
    input1=str(synth_root),         # synthetic dataset in current layout
    input2="cifar10-train",         # original CIFAR-10 train split as real reference
    cuda=True,
    isc=True,
    fid=True,
    kid=True,
    prc=True,
    batch_size=BATCH_SIZE,
    samples_find_deep=True,
    input1_cache_name="synthetic_cifar_like",
)

print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v}")


/usr/local/lib/python3.12/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']


CIFAR-10 downloaded under: cifar10_reference_data
Synthetic dataset path: /workspace/cifar32


Extracting features from input1
Loading cached /root/.cache/torch/fidelity_cache/synthetic_cifar_like-inception-v3-compat-features-logits_unbiased.pt
Loading cached /root/.cache/torch/fidelity_cache/synthetic_cifar_like-inception-v3-compat-features-2048.pt
Extracting features from input2
Loading cached /root/.cache/torch/fidelity_cache/cifar10-train-inception-v3-compat-features-logits_unbiased.pt
Loading cached /root/.cache/torch/fidelity_cache/cifar10-train-inception-v3-compat-features-2048.pt
Inception Score: 8.768589 ± 0.1064444
Loading cached /root/.cache/torch/fidelity_cache/synthetic_cifar_like-inception-v3-compat-stat-fid-2048.pt
Loading cached /root/.cache/torch/fidelity_cache/cifar10-train-inception-v3-compat-stat-fid-2048.pt
Frechet Inception Distance: 35.1747
Kernel Inception Distance: 0.01825312 ± 0.001021965                             
Creating feature extractor "vgg16" with features ['fc2_relu']
Extracting features from input1
Looking for samples recursively in "/workspa


=== Metrics ===
inception_score_mean: 8.768588696633243
inception_score_std: 0.10644443757948319
frechet_inception_distance: 35.1746997717857
kernel_inception_distance_mean: 0.018253118991851808
kernel_inception_distance_std: 0.0010219650625222076
precision: 0.24444591999053955
recall: 0.26260000467300415
f_score: 0.25319797126624183


Precision: 0.2444459
Recall: 0.2626
F-score: 0.253198


In [7]:

# ===== 5) Optional: classwise counts check before structuring =====
# This helps verify the source dataset is balanced before splitting.

from pprint import pprint
pprint(counts)
print("Min class count:", min(counts.values()))
print("Max class count:", max(counts.values()))


{'airplane': 6000,
 'automobile': 6000,
 'bird': 6000,
 'cat': 6000,
 'deer': 6000,
 'dog': 6000,
 'frog': 6000,
 'horse': 6000,
 'ship': 6000,
 'truck': 6000}
Min class count: 6000
Max class count: 6000


In [8]:

# ===== 6) Split dataset into CIFAR-10-style train/test structure (50k/10k total) =====
# Original CIFAR-10 has 5000 train and 1000 test images per class.
# Since your dataset is 60k balanced across 10 classes, we reproduce that:
#   - 5000 train per class
#   - 1000 test per class

random.seed(SEED)

out_root = Path(OUT_ROOT)
train_root = out_root / "train"
test_root = out_root / "test"

if out_root.exists():
    print(f"Removing existing output folder: {out_root}")
    shutil.rmtree(out_root)

for split_root in [train_root, test_root]:
    for c in CLASS_NAMES:
        (split_root / c).mkdir(parents=True, exist_ok=True)

summary = {}

for c in CLASS_NAMES:
    files = [p for p in (synth_root / c).iterdir() if p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".webp"}]
    files = sorted(files)
    random.shuffle(files)

    assert len(files) == 6000, f"Expected 6000 images in class {c}, found {len(files)}"

    train_files = files[:5000]
    test_files = files[5000:6000]

    for src in tqdm(train_files, desc=f"Copy train/{c}"):
        shutil.copy2(src, train_root / c / src.name)

    for src in tqdm(test_files, desc=f"Copy test/{c}"):
        shutil.copy2(src, test_root / c / src.name)

    summary[c] = {"train": len(train_files), "test": len(test_files)}

print("\nSplit summary:")
for c, s in summary.items():
    print(f"{c:>10}: train={s['train']}, test={s['test']}")


Copy train/airplane:   0%|          | 0/5000 [00:00<?, ?it/s]

Copy test/airplane:   0%|          | 0/1000 [00:00<?, ?it/s]

Copy train/automobile:   0%|          | 0/5000 [00:00<?, ?it/s]

Copy test/automobile:   0%|          | 0/1000 [00:00<?, ?it/s]

Copy train/bird:   0%|          | 0/5000 [00:00<?, ?it/s]

...

Copy train/ship:   0%|          | 0/5000 [00:00<?, ?it/s]

Copy test/ship:   0%|          | 0/1000 [00:00<?, ?it/s]

Copy train/truck:   0%|          | 0/5000 [00:00<?, ?it/s]

Copy test/truck:   0%|          | 0/1000 [00:00<?, ?it/s]


Split summary:
  airplane: train=5000, test=1000
automobile: train=5000, test=1000
      bird: train=5000, test=1000
       cat: train=5000, test=1000
      deer: train=5000, test=1000
       dog: train=5000, test=1000
      frog: train=5000, test=1000
     horse: train=5000, test=1000
      ship: train=5000, test=1000
     truck: train=5000, test=1000


In [9]:

# ===== 7) Verify final structure =====

def count_images(folder):
    return len([p for p in Path(folder).rglob("*") if p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".webp"}])

train_total = count_images(train_root)
test_total = count_images(test_root)

print("Train total:", train_total)
print("Test total :", test_total)
print("Grand total:", train_total + test_total)

assert train_total == 50000, f"Expected 50000 train images, found {train_total}"
assert test_total == 10000, f"Expected 10000 test images, found {test_total}"


Train total: 50000
Test total : 10000
Grand total: 60000


In [10]:

# ===== 8) Zip the structured dataset =====

zip_path = Path(ZIP_PATH)
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in tqdm(out_root.rglob("*"), desc="Zipping"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(out_root.parent))

print("Created zip:", zip_path.resolve())
print("Size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))


Zipping: 0it [00:00, ?it/s]

Created zip: /workspace/model_cache/CIFAR10_like_structured.zip
Size (MB): 147.73


In [11]:

# ===== 9) Save metrics to a text file =====

metrics_path = Path("fid_is_results.txt")
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("FID / Inception Score Results\n")
    f.write("============================\n")
    for k, v in metrics.items():
        f.write(f"{k}: {v}\n")

print("Saved:", metrics_path.resolve())


Saved: /workspace/model_cache/fid_is_results.txt


## Notes
- The notebook now computes **FID and Inception Score before any restructuring**.
- The split assumes **6000 images per class** so it can reproduce CIFAR-10's **5000 train / 1000 test per class**.
- FID uses **original CIFAR-10 train** as the real reference.
- Inception Score is computed on your current dataset as provided.
